<a href="https://colab.research.google.com/github/lkhar21/Homework4_ML/blob/main/homework4.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:

!pip install wandb -qU
!pip install kaggle -q

import os
import torch
import torchvision
import wandb

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(print(f"მუშაობთ მოწყობილობაზე: {device}"))

მუშაობთ მოწყობილობაზე: cuda
None


In [2]:
import wandb
wandb.login("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")

/usr/local/lib/python3.12/dist-packages/notebook/notebookapp.py:191: SyntaxWarning: invalid escape sequence '\/'
  | |_| | '_ \/ _` / _` |  _/ -_)
wandb: WARNING If you're specifying your api key in code, ensure this code is not shared publicly.
wandb: WARNING Consider setting the WANDB_API_KEY environment variable, or running `wandb login` from the command line.
wandb: [wandb.login()] Using explicit session credentials for https://api.wandb.ai.
wandb: No netrc file found, creating one.
wandb: Appending key for api.wandb.ai to your netrc file: /root/.netrc
wandb: Currently logged in as: lkhar21 (lkhar21-free-university-of-tbilisi-) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin


True

In [4]:
import os

!mkdir -p ~/.kaggle
!cp kaggle.json ~/.kaggle/
!chmod 600 ~/.kaggle/kaggle.json

In [5]:

!kaggle competitions download -c challenges-in-representation-learning-facial-expression-recognition-challenge

import zipfile

with zipfile.ZipFile("challenges-in-representation-learning-facial-expression-recognition-challenge.zip", "r") as zip_ref:
    zip_ref.extractall("fer2013_data")

print("მონაცემები წარმატებით ჩამოიტვირთა და დაარქივდა!")

100% 285M/285M [00:02<00:00, 139MB/s]

მონაცემები წარმატებით ჩამოიტვირთა და დაარქივდა!


In [7]:
os.listdir("fer2013_data")

['example_submission.csv',
 'fer2013.tar.gz',
 'icml_face_data.csv',
 'test.csv',
 'train.csv']

In [8]:
import os
print(os.listdir("fer2013_data"))

['example_submission.csv', 'fer2013.tar.gz', 'icml_face_data.csv', 'test.csv', 'train.csv']


In [14]:
import pandas as pd
import numpy as np
import torch
from torch.utils.data import Dataset, DataLoader
import torchvision.transforms as transforms
from PIL import Image

class FER2013Dataset(Dataset):
    def __init__(self, csv_file, transform=None):
        self.df = pd.read_csv(csv_file)
        self.transform = transform

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]

        label = int(row['emotion']) if 'emotion' in row else 0

        pixels = np.fromstring(row['pixels'], dtype=int, sep=' ')
        pixels = pixels.reshape(48, 48).astype(np.uint8)

        image = Image.fromarray(pixels)

        if self.transform:
            image = self.transform(image)

        return image, label

train_transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize((0.5,), (0.5,))
])

val_transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize((0.5,), (0.5,))
])
train_transform_augmented = transforms.Compose([
    transforms.RandomHorizontalFlip(p=0.5),
    transforms.RandomRotation(15),
    transforms.ToTensor(),
    transforms.Normalize((0.5,), (0.5,))
])

train_dataset_aug = FER2013Dataset(csv_file=train_path, transform=train_transform_augmented)
train_loader_aug = DataLoader(train_dataset_aug, batch_size=64, shuffle=True)

train_path = 'fer2013_data/train.csv'
test_path = 'fer2013_data/test.csv'

train_dataset = FER2013Dataset(csv_file=train_path, transform=train_transform)
val_dataset = FER2013Dataset(csv_file=test_path, transform=val_transform)

train_loader = DataLoader(train_dataset, batch_size=64, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=64, shuffle=False)

print(f"Train სემპლები: {len(train_dataset)}, Validation სემპლები: {len(val_dataset)}")

Train სემპლები: 28709, Validation სემპლები: 7178


In [10]:
import torch.nn as nn
import torch.nn.functional as F

class MiniCNN(nn.Module):
    def __init__(self):
        super(MiniCNN, self).__init__()
        self.conv1 = nn.Conv2d(1, 16, kernel_size=3, padding=1)
        self.pool = nn.MaxPool2d(2, 2)
        self.fc1 = nn.Linear(16 * 24 * 24, 7)

    def forward(self, x):
        x = self.pool(F.relu(self.conv1(x)))
        x = x.view(-1, 16 * 24 * 24)
        x = self.fc1(x)
        return x

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = MiniCNN().to(device)
criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=0.001)

data_iter = iter(train_loader)
images, labels = next(data_iter)
images, labels = images.to(device), labels.to(device)

print("იწყება Sanity Check 1 ბატჩზე...")
for epoch in range(60):
    optimizer.zero_grad()
    outputs = model(images)
    loss = criterion(outputs, labels)
    loss.backward()
    optimizer.step()

    if (epoch + 1) % 10 == 0:
        _, predicted = torch.max(outputs.data, 1)
        acc = (predicted == labels).sum().item() / labels.size(0) * 100
        print(f"Epoch [{epoch+1}/60], Loss: {loss.item():.4f}, Accuracy: {acc:.2f}%")

იწყება Sanity Check 1 ბატჩზე...
Epoch [10/60], Loss: 0.7710, Accuracy: 82.81%
Epoch [20/60], Loss: 0.2433, Accuracy: 98.44%
Epoch [30/60], Loss: 0.0751, Accuracy: 100.00%
Epoch [40/60], Loss: 0.0325, Accuracy: 100.00%
Epoch [50/60], Loss: 0.0185, Accuracy: 100.00%
Epoch [60/60], Loss: 0.0129, Accuracy: 100.00%


In [11]:
import wandb

def train_and_log(model, train_loader, val_loader, config):
    wandb.init(project="fer2013-facial-expression", config=config, name=config['run_name'])

    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    model = model.to(device)

    criterion = nn.CrossEntropyLoss()

    if config['optimizer'] == 'Adam':
        optimizer = torch.optim.Adam(model.parameters(), lr=config['lr'], weight_decay=config['weight_decay'])
    else:
        optimizer = torch.optim.SGD(model.parameters(), lr=config['lr'], momentum=0.9, weight_decay=config['weight_decay'])

    epochs = config['epochs']

    print(f"🚀 იწყება ექსპერიმენტი: {config['run_name']}")

    for epoch in range(epochs):
        # --- TRAINING PHASE ---
        model.train()
        running_loss = 0.0
        correct = 0
        total = 0

        for images, labels in train_loader:
            images, labels = images.to(device), labels.to(device)

            optimizer.zero_grad()
            outputs = model(images)
            loss = criterion(outputs, labels)
            loss.backward()
            optimizer.step()

            running_loss += loss.item() * images.size(0)
            _, predicted = torch.max(outputs.data, 1)
            total += labels.size(0)
            correct += (predicted == labels).sum().item()

        train_loss = running_loss / len(train_loader.dataset)
        train_acc = (correct / total) * 100

        # --- VALIDATION PHASE ---
        model.eval()
        val_loss = 0.0
        val_correct = 0
        val_total = 0

        with torch.no_grad():
            for images, labels in val_loader:
                images, labels = images.to(device), labels.to(device)
                outputs = model(images)
                loss = criterion(outputs, labels)

                val_loss += loss.item() * images.size(0)
                _, predicted = torch.max(outputs.data, 1)
                val_total += labels.size(0)
                val_correct += (predicted == labels).sum().item()

        valid_loss = val_loss / len(val_loader.dataset)
        valid_acc = (val_correct / val_total) * 100

        wandb.log({
            "epoch": epoch + 1,
            "train_loss": train_loss,
            "train_accuracy": train_acc,
            "val_loss": valid_loss,
            "val_accuracy": valid_acc
        })

        print(f"Epoch [{epoch+1}/{epochs}] | Train Loss: {train_loss:.4f}, Train Acc: {train_acc:.2f}% | Val Loss: {valid_loss:.4f}, Val Acc: {valid_acc:.2f}%")

    wandb.finish()

In [15]:
class BaselineCNN(nn.Module):
    def __init__(self):
        super(BaselineCNN, self).__init__()
        self.conv1 = nn.Conv2d(1, 32, kernel_size=3, padding=1)
        self.conv2 = nn.Conv2d(32, 64, kernel_size=3, padding=1)
        self.pool = nn.MaxPool2d(2, 2)

        self.fc1 = nn.Linear(64 * 12 * 12, 128)
        self.fc2 = nn.Linear(128, 7)

    def forward(self, x):
        x = self.pool(F.relu(self.conv1(x)))
        x = self.pool(F.relu(self.conv2(x)))
        x = x.view(-1, 64 * 12 * 12)
        x = F.relu(self.fc1(x))
        x = self.fc2(x)
        return x

config_baseline = {
    "run_name": "Architecture_1_Baseline",
    "lr": 0.01,
    "optimizer": "SGD",
    "epochs": 10,
    "weight_decay": 0.0
}

train_and_log(BaselineCNN(), train_loader, val_loader, config_baseline)

🚀 იწყება ექსპერიმენტი: Architecture_1_Baseline
Epoch [1/10] | Train Loss: 1.6800, Train Acc: 32.98% | Val Loss: 2.1071, Val Acc: 2.22%
Epoch [2/10] | Train Loss: 1.4849, Train Acc: 42.98% | Val Loss: 2.0519, Val Acc: 13.47%
Epoch [3/10] | Train Loss: 1.3502, Train Acc: 48.33% | Val Loss: 2.2890, Val Acc: 15.39%
Epoch [4/10] | Train Loss: 1.2540, Train Acc: 52.06% | Val Loss: 2.4834, Val Acc: 13.82%
Epoch [5/10] | Train Loss: 1.1603, Train Acc: 55.98% | Val Loss: 3.2656, Val Acc: 6.51%
Epoch [6/10] | Train Loss: 1.0598, Train Acc: 60.34% | Val Loss: 3.0186, Val Acc: 11.59%
Epoch [7/10] | Train Loss: 0.9448, Train Acc: 65.06% | Val Loss: 3.6003, Val Acc: 11.80%
Epoch [8/10] | Train Loss: 0.8099, Train Acc: 70.32% | Val Loss: 3.2975, Val Acc: 16.16%
Epoch [9/10] | Train Loss: 0.6658, Train Acc: 76.34% | Val Loss: 4.4889, Val Acc: 12.84%
Epoch [10/10] | Train Loss: 0.5048, Train Acc: 82.37% | Val Loss: 4.7725, Val Acc: 12.20%


epoch,▁▂▃▃▄▅▆▆▇█
train_accuracy,▁▂▃▄▄▅▆▆▇█
train_loss,█▇▆▅▅▄▄▃▂▁
val_accuracy,▁▇█▇▃▆▆█▆▆
val_loss,▁▁▂▂▄▃▅▄▇█
epoch,10
train_accuracy,82.37487
train_loss,0.50478
val_accuracy,12.20396
val_loss,4.7725


In [12]:
import torch.nn as nn
import torch.nn.functional as F

class BaselineCNNDropout(nn.Module):
    def __init__(self):
        super(BaselineCNNDropout, self).__init__()
        self.conv1 = nn.Conv2d(1, 32, kernel_size=3, padding=1)
        self.conv2 = nn.Conv2d(32, 64, kernel_size=3, padding=1)
        self.pool = nn.MaxPool2d(2, 2)

        self.dropout1 = nn.Dropout(0.25)
        self.dropout2 = nn.Dropout(0.5)

        self.fc1 = nn.Linear(64 * 12 * 12, 128)
        self.fc2 = nn.Linear(128, 7)

    def forward(self, x):
        x = self.pool(F.relu(self.conv1(x)))
        x = self.pool(F.relu(self.conv2(x)))
        x = self.dropout1(x)

        x = x.view(-1, 64 * 12 * 12)
        x = F.relu(self.fc1(x))
        x = self.dropout2(x)
        x = self.fc2(x)
        return x

config_experiment_2 = {
    "run_name": "Architecture_2_Baseline_With_Dropout",
    "lr": 0.001,
    "optimizer": "Adam",
    "epochs": 10,
    "weight_decay": 1e-4
}

train_and_log(BaselineCNNDropout(), train_loader, val_loader, config_experiment_2)

🚀 იწყება ექსპერიმენტი: Architecture_2_Baseline_With_Dropout
Epoch [1/10] | Train Loss: 1.6547, Train Acc: 34.15% | Val Loss: 2.1548, Val Acc: 3.08%
Epoch [2/10] | Train Loss: 1.4638, Train Acc: 43.11% | Val Loss: 2.2304, Val Acc: 8.97%
Epoch [3/10] | Train Loss: 1.3844, Train Acc: 46.90% | Val Loss: 2.3837, Val Acc: 9.45%
Epoch [4/10] | Train Loss: 1.3304, Train Acc: 48.97% | Val Loss: 2.3538, Val Acc: 13.64%
Epoch [5/10] | Train Loss: 1.2856, Train Acc: 50.50% | Val Loss: 2.6741, Val Acc: 9.14%
Epoch [6/10] | Train Loss: 1.2399, Train Acc: 52.70% | Val Loss: 2.5268, Val Acc: 11.83%
Epoch [7/10] | Train Loss: 1.2010, Train Acc: 54.08% | Val Loss: 2.9662, Val Acc: 7.63%
Epoch [8/10] | Train Loss: 1.1729, Train Acc: 55.39% | Val Loss: 2.8895, Val Acc: 8.90%
Epoch [9/10] | Train Loss: 1.1327, Train Acc: 56.84% | Val Loss: 2.9605, Val Acc: 12.15%
Epoch [10/10] | Train Loss: 1.1050, Train Acc: 57.80% | Val Loss: 3.0211, Val Acc: 11.67%


epoch,▁▂▃▃▄▅▆▆▇█
train_accuracy,▁▄▅▅▆▆▇▇██
train_loss,█▆▅▄▃▃▂▂▁▁
val_accuracy,▁▅▅█▅▇▄▅▇▇
val_loss,▁▂▃▃▅▄█▇██
epoch,10
train_accuracy,57.80417
train_loss,1.10503
val_accuracy,11.67456
val_loss,3.02112


In [13]:
class OptimizedDeepCNN(nn.Module):
    def __init__(self):
        super(OptimizedDeepCNN, self).__init__()

        self.features = nn.Sequential(

            nn.Conv2d(1, 32, kernel_size=3, padding=1),
            nn.BatchNorm2d(32),
            nn.ReLU(),
            nn.Conv2d(32, 64, kernel_size=3, padding=1),
            nn.BatchNorm2d(64),
            nn.ReLU(),
            nn.MaxPool2d(2, 2),
            nn.Dropout(0.25),

            nn.Conv2d(64, 128, kernel_size=3, padding=1),
            nn.BatchNorm2d(128),
            nn.ReLU(),
            nn.MaxPool2d(2, 2),
            nn.Dropout(0.25),
        )

        self.classifier = nn.Sequential(
            nn.Linear(128 * 12 * 12, 256),
            nn.BatchNorm1d(256),
            nn.ReLU(),
            nn.Dropout(0.5),
            nn.Linear(256, 7)
        )

    def forward(self, x):
        x = self.features(x)
        x = x.view(x.size(0), -1)
        x = self.classifier(x)
        return x

config_experiment_3 = {
    "run_name": "Architecture_3_Optimized_DeepCNN",
    "lr": 0.001,
    "optimizer": "Adam",
    "epochs": 10,
    "weight_decay": 1e-4
}

train_and_log(OptimizedDeepCNN(), train_loader, val_loader, config_experiment_3)

🚀 იწყება ექსპერიმენტი: Architecture_3_Optimized_DeepCNN
Epoch [1/10] | Train Loss: 1.4892, Train Acc: 42.55% | Val Loss: 2.4221, Val Acc: 7.90%
Epoch [2/10] | Train Loss: 1.2716, Train Acc: 51.26% | Val Loss: 2.7118, Val Acc: 10.45%
Epoch [3/10] | Train Loss: 1.1763, Train Acc: 55.16% | Val Loss: 2.7979, Val Acc: 10.98%
Epoch [4/10] | Train Loss: 1.1037, Train Acc: 58.40% | Val Loss: 2.8191, Val Acc: 13.42%
Epoch [5/10] | Train Loss: 1.0321, Train Acc: 61.50% | Val Loss: 2.9833, Val Acc: 13.69%
Epoch [6/10] | Train Loss: 0.9650, Train Acc: 64.13% | Val Loss: 3.4330, Val Acc: 10.45%
Epoch [7/10] | Train Loss: 0.8951, Train Acc: 66.90% | Val Loss: 3.1588, Val Acc: 13.35%
Epoch [8/10] | Train Loss: 0.8212, Train Acc: 69.76% | Val Loss: 3.8064, Val Acc: 9.67%
Epoch [9/10] | Train Loss: 0.7607, Train Acc: 71.93% | Val Loss: 3.3834, Val Acc: 16.73%
Epoch [10/10] | Train Loss: 0.7056, Train Acc: 74.37% | Val Loss: 3.5331, Val Acc: 13.82%


epoch,▁▂▃▃▄▅▆▆▇█
train_accuracy,▁▃▄▄▅▆▆▇▇█
train_loss,█▆▅▅▄▃▃▂▁▁
val_accuracy,▁▃▃▅▆▃▅▂█▆
val_loss,▁▂▃▃▄▆▅█▆▇
epoch,10
train_accuracy,74.36692
train_loss,0.70563
val_accuracy,13.82001
val_loss,3.53311


In [15]:
config_3a = {
    "run_name": "Architecture_3A_Optimized_With_Augmentation",
    "lr": 0.001,
    "optimizer": "Adam",
    "epochs": 10,
    "weight_decay": 1e-4
}
train_and_log(OptimizedDeepCNN(), train_loader_aug, val_loader, config_3a)

🚀 იწყება ექსპერიმენტი: Architecture_3A_Optimized_With_Augmentation
Epoch [1/10] | Train Loss: 1.5667, Train Acc: 38.91% | Val Loss: 2.5676, Val Acc: 4.56%
Epoch [2/10] | Train Loss: 1.3784, Train Acc: 46.91% | Val Loss: 2.4645, Val Acc: 15.31%
Epoch [3/10] | Train Loss: 1.3197, Train Acc: 49.59% | Val Loss: 2.8021, Val Acc: 11.54%
Epoch [4/10] | Train Loss: 1.2738, Train Acc: 51.14% | Val Loss: 3.1514, Val Acc: 7.94%
Epoch [5/10] | Train Loss: 1.2443, Train Acc: 52.73% | Val Loss: 2.9749, Val Acc: 8.44%
Epoch [6/10] | Train Loss: 1.2193, Train Acc: 53.80% | Val Loss: 2.9267, Val Acc: 8.94%
Epoch [7/10] | Train Loss: 1.1983, Train Acc: 54.41% | Val Loss: 3.1180, Val Acc: 9.07%
Epoch [8/10] | Train Loss: 1.1789, Train Acc: 55.34% | Val Loss: 3.0409, Val Acc: 9.42%
Epoch [9/10] | Train Loss: 1.1620, Train Acc: 55.75% | Val Loss: 2.8218, Val Acc: 14.47%
Epoch [10/10] | Train Loss: 1.1448, Train Acc: 56.72% | Val Loss: 3.0904, Val Acc: 9.56%


epoch,▁▂▃▃▄▅▆▆▇█
train_accuracy,▁▄▅▆▆▇▇▇██
train_loss,█▅▄▃▃▂▂▂▁▁
val_accuracy,▁█▆▃▄▄▄▄▇▄
val_loss,▂▁▄█▆▆█▇▅▇
epoch,10
train_accuracy,56.72437
train_loss,1.14481
val_accuracy,9.55698
val_loss,3.09045


In [16]:

def train_and_log_adamw(model, train_loader, val_loader, config):
    wandb.init(project="fer2013-facial-expression", config=config, name=config['run_name'])
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    model = model.to(device)
    criterion = nn.CrossEntropyLoss()

    optimizer = torch.optim.AdamW(model.parameters(), lr=config['lr'], weight_decay=config['weight_decay'])

    for epoch in range(config['epochs']):
        model.train()
        running_loss = 0.0
        correct = 0
        total = 0
        for images, labels in train_loader:
            images, labels = images.to(device), labels.to(device)
            optimizer.zero_grad()
            outputs = model(images)
            loss = criterion(outputs, labels)
            loss.backward()
            optimizer.step()
            running_loss += loss.item() * images.size(0)
            _, predicted = torch.max(outputs.data, 1)
            total += labels.size(0)
            correct += (predicted == labels).sum().item()

        train_loss = running_loss / len(train_loader.dataset)
        train_acc = (correct / total) * 100

        model.eval()
        val_loss = 0.0
        val_correct = 0
        val_total = 0
        with torch.no_grad():
            for images, labels in val_loader:
                images, labels = images.to(device), labels.to(device)
                outputs = model(images)
                loss = criterion(outputs, labels)
                val_loss += loss.item() * images.size(0)
                _, predicted = torch.max(outputs.data, 1)
                val_total += labels.size(0)
                val_correct += (predicted == labels).sum().item()

        valid_loss = val_loss / len(val_loader.dataset)
        valid_acc = (val_correct / val_total) * 100

        wandb.log({
            "epoch": epoch + 1,
            "train_loss": train_loss,
            "train_accuracy": train_acc,
            "val_loss": valid_loss,
            "val_accuracy": valid_acc
        })
    wandb.finish()

config_3b = {
    "run_name": "Architecture_3B_Optimized_AdamW_LowLR",
    "lr": 0.0001,
    "optimizer": "AdamW",
    "epochs": 10,
    "weight_decay": 1e-3
}
train_and_log_adamw(OptimizedDeepCNN(), train_loader_aug, val_loader, config_3b)

epoch,▁▂▃▃▄▅▆▆▇█
train_accuracy,▁▄▅▆▆▇▇███
train_loss,█▅▄▃▃▂▂▂▁▁
val_accuracy,▁▄▅▅▄▆▆▅▆█
val_loss,▁▄▆▆▇▆▇▇█▆
epoch,10
train_accuracy,52.41562
train_loss,1.25223
val_accuracy,9.90527
val_loss,2.88554
